# CAIR sentiment — RoBERTa-large, 5 folds, Kaggle T4

Ноутбук обучает Twitter-RoBERTa на `CAIRLAB.csv`, проверяет качество с разбиением по `ID` и сохраняет готовый `submission.csv`. Валидационная метрика считается только на первой (оригинальной) строке каждого ID — как и тест, она не содержит перефразированные варианты.

## Перед запуском

1. В Kaggle добавьте dataset с файлами `CAIRLAB.csv` и `test.csv`.
2. Включите **GPU: T4** и **Internet** (нужен доступ к Hugging Face для загрузки модели).
3. Запустите ячейки сверху вниз. Итоговый файл появится в `/kaggle/working/submission.csv`.

По текущему локальному CV это воспроизводимый сильный одиночный вариант, но он не доказанно лучше `sub_final` (76,125% на Kaggle). Он не обучает Qwen 8B: в текущих логах один fold Qwen занимал около трёх часов на T4, поэтому все пять folds Qwen на одной карте — непрактичный запуск.


In [ ]:
%pip -q install "transformers>=4.41,<5" scikit-learn


In [ ]:
import gc
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

# Kaggle T4-friendly defaults. Set MAX_EPOCHS=1 for a faster run.
SEED = 42
MODEL_NAME = "cardiffnlp/twitter-roberta-large-topic-sentiment-latest"
N_SPLITS = 5
MAX_EPOCHS = 2
EARLY_STOP_PATIENCE = 1
MAX_LENGTH = 128
BATCH_SIZE = 8
GRAD_ACCUM = 2
LEARNING_RATE = 1e-5
NUM_WORKERS = 2

assert torch.cuda.is_available(), "Включите GPU T4 в настройках Kaggle Notebook."
DEVICE = torch.device("cuda")
USE_AMP = True
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


In [ ]:
def resolve_input(filename):
    roots = [Path("/kaggle/input"), Path.cwd(), Path("/kaggle/working")]
    for root in roots:
        if root.exists():
            matches = sorted(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Не найден {filename}; добавьте файлы в Kaggle Input.")

TRAIN_PATH = resolve_input("CAIRLAB.csv")
TEST_PATH = TRAIN_PATH.parent / "test.csv"
if not TEST_PATH.exists():
    TEST_PATH = resolve_input("test.csv")
print("Train:", TRAIN_PATH)
print("Test: ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH, header=None, names=["ID", "Entity", "Sentiment", "Text"])
test_df = pd.read_csv(TEST_PATH)
required_test = {"ID", "Entity", "Text"}
if not required_test.issubset(test_df.columns):
    raise ValueError(f"В test.csv должны быть колонки {sorted(required_test)}")

for df in (train_df, test_df):
    df["Entity"] = df["Entity"].fillna("").astype(str)
    df["Text"] = df["Text"].fillna("").astype(str)
train_df = train_df[train_df["Text"].str.strip().ne("")].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
if train_df["ID"].isna().any() or test_df["ID"].isna().any():
    raise ValueError("Обнаружены пустые ID.")
if train_df.groupby("ID")["Sentiment"].nunique().max() != 1:
    raise ValueError("У одного ID встречаются разные sentiment-метки.")
overlap = set(train_df["ID"]) & set(test_df["ID"])
if overlap:
    raise ValueError(f"Train/test ID пересекаются ({len(overlap)} ID); проверьте входные файлы.")

# Первая строка группы — исходный текст; поздние варианты иногда сильно укорочены.
train_df["pos"] = train_df.groupby("ID").cumcount()
first_len = train_df.groupby("ID")["Text"].transform("first").str.len().clip(lower=1)
length_ratio = train_df["Text"].str.len() / first_len
is_original = train_df["pos"].eq(0).to_numpy()
keep_aug = ((train_df["pos"] < 5) & (length_ratio > 0.5)).to_numpy()

classes = np.array(sorted(train_df["Sentiment"].unique()))
label_to_id = {label: i for i, label in enumerate(classes)}
y = train_df["Sentiment"].map(label_to_id).to_numpy(dtype=np.int64)
groups = train_df["ID"].to_numpy()
splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(splitter.split(np.zeros(len(train_df)), y, groups=groups))

print("Train rows:", len(train_df), "| test rows:", len(test_df))
print("Unique train IDs:", train_df["ID"].nunique(), "| retained train variants:", int(keep_aug.sum()))
print("Labels:", dict(zip(*np.unique(train_df["Sentiment"], return_counts=True))))
print("Classes:", classes.tolist())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
train_enc = tokenizer(
    train_df["Entity"].tolist(), train_df["Text"].tolist(),
    truncation=True, max_length=MAX_LENGTH, padding=False,
)
test_enc = tokenizer(
    test_df["Entity"].tolist(), test_df["Text"].tolist(),
    truncation=True, max_length=MAX_LENGTH, padding=False,
)

class EncodedDataset(Dataset):
    def __init__(self, encodings, indices, labels=None):
        self.encodings = encodings
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = labels
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, item):
        idx = int(self.indices[item])
        row = {key: self.encodings[key][idx] for key in self.encodings.keys()}
        if self.labels is not None:
            row["labels"] = int(self.labels[idx])
        return row

def collate_batch(rows):
    labels = None
    if "labels" in rows[0]:
        labels = torch.tensor([row.pop("labels") for row in rows], dtype=torch.long)
    batch = tokenizer.pad(rows, padding=True, pad_to_multiple_of=8, return_tensors="pt")
    if labels is not None:
        batch["labels"] = labels
    return batch

def make_loader(dataset, batch_size, shuffle=False):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_batch,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0),
    )

print("Tokenized train/test rows:", len(train_enc["input_ids"]), len(test_enc["input_ids"]))


In [ ]:
def evaluate(model, loader):
    model.eval()
    all_probs, all_y, losses = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(**batch, labels=labels)
            losses.append(float(out.loss.item()))
            all_probs.append(out.logits.float().softmax(-1).cpu().numpy())
            all_y.append(labels.cpu().numpy())
    probs = np.concatenate(all_probs)
    gold = np.concatenate(all_y)
    return float(np.mean(losses)), float((probs.argmax(1) == gold).mean()), probs

@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(**batch).logits
        all_probs.append(logits.float().softmax(-1).cpu().numpy())
    return np.concatenate(all_probs)

oof_probs = np.full((len(train_df), len(classes)), np.nan, dtype=np.float32)
test_probs_folds = []
fold_metrics = []
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
checkpoint_path = OUT_DIR / "roberta_t4_best_fold.pt"

for fold_id, (tr_idx, va_idx) in enumerate(folds):
    print(f"\n===== Fold {fold_id + 1}/{N_SPLITS} =====")
    seed_everything(SEED + fold_id)
    tr_idx = tr_idx[keep_aug[tr_idx]]
    va_original_idx = va_idx[is_original[va_idx]]
    if len(va_original_idx) == 0:
        raise RuntimeError(f"В fold {fold_id} нет original validation rows.")

    train_loader = make_loader(EncodedDataset(train_enc, tr_idx, y), BATCH_SIZE, shuffle=True)
    val_loader = make_loader(EncodedDataset(train_enc, va_original_idx, y), BATCH_SIZE)
    test_loader = make_loader(EncodedDataset(test_enc, np.arange(len(test_df))), BATCH_SIZE)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(classes), ignore_mismatched_sizes=True,
        id2label={i: str(label) for i, label in enumerate(classes)},
        label2id={str(label): i for i, label in enumerate(classes)},
    ).to(DEVICE)
    model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    total_steps = updates_per_epoch * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(0.06 * total_steps)),
        num_training_steps=total_steps,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    best_acc, best_loss, best_epoch, bad_epochs = -1.0, float("inf"), 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        progress = tqdm(train_loader, desc=f"fold {fold_id} epoch {epoch + 1}")
        for step, batch in enumerate(progress):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            group_start = (step // GRAD_ACCUM) * GRAD_ACCUM
            group_size = min(GRAD_ACCUM, len(train_loader) - group_start)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                loss = model(**batch).loss / group_size
            scaler.scale(loss).backward()
            running_loss += float(loss.item()) * group_size
            should_step = ((step + 1) % GRAD_ACCUM == 0) or (step + 1 == len(train_loader))
            if should_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            if step % 100 == 0:
                progress.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        val_loss, val_acc, _ = evaluate(model, val_loader)
        print(f"fold {fold_id} | epoch {epoch + 1} | original validation accuracy={val_acc:.4f} | loss={val_loss:.4f}")
        improved = (val_acc > best_acc) or (math.isclose(val_acc, best_acc) and val_loss < best_loss)
        if improved:
            best_acc, best_loss, best_epoch, bad_epochs = val_acc, val_loss, epoch + 1, 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            bad_epochs += 1
            if bad_epochs >= EARLY_STOP_PATIENCE:
                print("Early stopping")
                break

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    val_loss, val_acc, val_fold_probs = evaluate(model, val_loader)
    test_fold_probs = predict(model, test_loader)
    oof_probs[va_original_idx] = val_fold_probs
    test_probs_folds.append(test_fold_probs)
    fold_metrics.append({"fold": fold_id, "best_epoch": best_epoch, "val_accuracy": val_acc, "val_loss": val_loss})
    print(f"Restored best epoch {best_epoch}; fold accuracy={val_acc:.4f}")

    del model, optimizer, scheduler, scaler, train_loader, val_loader, test_loader
    if checkpoint_path.exists():
        checkpoint_path.unlink()
    gc.collect()
    torch.cuda.empty_cache()

    # Save progress after every completed fold.
    np.save(OUT_DIR / "roberta_t4_oof_progress.npy", oof_probs)
    np.save(OUT_DIR / "roberta_t4_test_folds_progress.npy", np.stack(test_probs_folds))


In [ ]:
valid_original = is_original & np.isfinite(oof_probs).all(axis=1)
cv_accuracy = float((oof_probs[valid_original].argmax(1) == y[valid_original]).mean())
test_probs = np.mean(test_probs_folds, axis=0)
pred_labels = classes[test_probs.argmax(axis=1)]

submission = pd.DataFrame({"ID": test_df["ID"], "Sentiment": pred_labels})
assert len(submission) == len(test_df)
assert submission["ID"].is_unique
assert set(submission["Sentiment"]).issubset(set(classes))
submission.to_csv(OUT_DIR / "submission.csv", index=False)

print(pd.DataFrame(fold_metrics).to_string(index=False))
print(f"Original-row OOF accuracy: {cv_accuracy:.4f} on {valid_original.sum()} rows")
print("Saved:", OUT_DIR / "submission.csv")
display(submission.head())
display(submission["Sentiment"].value_counts(normalize=True).rename("share").to_frame())


## Что делает валидация

Все варианты одного `ID` остаются в одном fold, чтобы почти одинаковые тексты не попали одновременно в train и validation. Для обучения оставляются исходный текст и до четырёх достаточно длинных вариантов; validation и итоговая оценка берутся только по исходным текстам. На каждом fold сохраняется эпоха с лучшей original-row accuracy.

OOF accuracy здесь используется для выбора эпохи, поэтому её следует считать ориентиром, а итоговый Kaggle score — окончательным сравнением. Для честного сравнения с `sub_final.csv` отправляйте submission отдельно и сравнивайте score на том же leaderboard.
